In [ ]:
import gc
import os

import numpy as np
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger
from torch.utils.data import DataLoader

from benchmarks import (
    BestEpochTracker,
    LightningModule,
    MultiTargetScaler,
    PrintEpochMetricsCallback,
)
from dataset.pac_matrix import PAC5MatrixDataset
from gcnn.base_model import GroupEquivariantCNN

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
FOLDS = range(1, k_folds + 1)

target_properties = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "extensive",
]

TAG = "gcnn_1.1M-pac_grid"
model_path_template = "../models/" + TAG + "-[PAC5-4]-fold_{fold}"
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = TAG + "-[PAC5-4]-fold_{fold}"

SPLIT_DIR = model_dataset_dir + "/PAC5-4"
NUM_WORKERS = 12
KERNEL_SIZE = 5
NUM_HIDDEN = 3
HIDDEN_CHANNELS = 32
BATCH_SIZE = 256
MAX_EPOCHS = 500
ES_PATIENCE = 50
LR_PATIENCE = 25
MIN_LR = 1e-6
LR = 1e-3


def _plateau_optimizers(self):
    """Configure AdamW with ReduceLROnPlateau in place of OneCycleLR."""
    optimizer = torch.optim.AdamW(
        self.parameters(), lr=self.lr, weight_decay=self.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=LR_PATIENCE, min_lr=MIN_LR
    )
    return {
        "optimizer": optimizer,
        "lr_scheduler": {
            "scheduler": scheduler,
            "monitor": "val_loss",
            "interval": "epoch",
            "frequency": 1,
        },
    }


class PlateauLightningModule(LightningModule):
    """LightningModule from benchmarks with ReduceLROnPlateau instead of OneCycleLR."""

    configure_optimizers = _plateau_optimizers


def build_model():
    return GroupEquivariantCNN(
        property_types=property_types,
        kernel_size=KERNEL_SIZE,
        num_hidden=NUM_HIDDEN,
        hidden_channels=HIDDEN_CHANNELS,
    )


def load_split(fold, part, scaler=None):
    dataset = PAC5MatrixDataset(
        SPLIT_DIR + f"/fold_{fold}_{part}.csv",
        properties=target_properties,
        num_workers=NUM_WORKERS,
    )
    if scaler is None:
        scaler = MultiTargetScaler()
        scaler.fit(dataset.targets.numpy())
    dataset.targets = torch.tensor(
        scaler.transform(dataset.targets.numpy()), dtype=torch.float32
    )
    return dataset, scaler


_m = build_model()
print(
    f"PAC G-CNN: depth {NUM_HIDDEN}, {HIDDEN_CHANNELS} hidden channels, "
    f"{KERNEL_SIZE}x{KERNEL_SIZE} kernel"
)
print(
    f"Trainable parameters: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}"
)
del _m
gc.collect()

In [ ]:
# -----------------------------------------------------------------------------
# 1. Training Loop
# -----------------------------------------------------------------------------
fold_results = []
best_epochs = []
stopped_epochs = []

for fold in FOLDS:
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_dataset, scaler = load_split(fold, "train")
    val_dataset, _ = load_split(fold, "val", scaler)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = build_model()
    print(
        f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
    )
    lightning_model = PlateauLightningModule(model, target_properties, scaler, lr=LR)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )
    early_stop = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=ES_PATIENCE,
        min_delta=0.0,
        check_on_train_epoch_end=False,
        verbose=False,
    )
    best_epoch_tracker = BestEpochTracker()

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[
            checkpoint_callback,
            early_stop,
            best_epoch_tracker,
            PrintEpochMetricsCallback(target_properties),
        ],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    best_epoch, last_epoch = best_epoch_tracker.best_epoch, best_epoch_tracker.last_epoch
    if best_epoch is None:
        raise RuntimeError("The best-epoch tracker did not record a validation epoch.")
    best_epochs.append(best_epoch)
    stopped_epochs.append(last_epoch)
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_lightning_model = PlateauLightningModule.load_from_checkpoint(
        checkpoint_callback.best_model_path,
        model=model,
        property_names=target_properties,
        scaler=scaler,
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_dataset, val_dataset, train_loader, val_loader
    del model, lightning_model, best_lightning_model
    del trainer, checkpoint_callback, logger, early_stop, best_epoch_tracker, scaler

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"\nBest-validation epoch per fold: {best_epochs} (ran {stopped_epochs})")
print(
    f"Latest best-validation epoch: {max(best_epochs)} of a {MAX_EPOCHS}-epoch ceiling"
)

In [ ]:
# -----------------------------------------------------------------------------
# 2. Evaluation Loop
# -----------------------------------------------------------------------------
import numpy as np

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold in FOLDS:
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_dataset, scaler = load_split(fold, "train")
    test_dataset, _ = load_split(fold, "test", scaler)

    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

    model = build_model()
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = PlateauLightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("\n======[ Train Set Evaluation ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("\n======[ Test Set Evaluation ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        for split, res in (("train", train_results), ("test", test_results)):
            mae = res[f"test_mae_{prop}"]
            rmsd = res[f"test_rmsd_{prop}"]
            mx = res[f"test_max_{prop}"]
            mn = res[f"test_min_{prop}"]
            r2 = res[f"test_r2_{prop}"]
            label = "Train" if split == "train" else "Test"
            print(
                f" + {label} -> MAE: {mae:.4f}\tRMSD: {rmsd:.4f}\tMax: {mx:.4f}\tMin: {mn:.4f}\tR^2: {r2:.4f}"
            )
            all_results[split][prop]["mae"].append(mae)
            all_results[split][prop]["rmsd"].append(rmsd)
            all_results[split][prop]["max"].append(mx)
            all_results[split][prop]["min"].append(mn)
            all_results[split][prop]["r2"].append(r2)

    del model, lightning_model, trainer, scaler, train_results, test_results
    del train_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n======[ Cross-validation results: {TAG} ]======")

ddof = 1 if len(list(FOLDS)) > 1 else 0
for prop in target_properties:
    print(f"\n======[ {prop} ]======")
    for split in ("train", "test"):
        st = {
            m: (
                np.mean(all_results[split][prop][m]),
                np.std(all_results[split][prop][m], ddof=ddof),
            )
            for m in metric_keys
        }
        label = "Train" if split == "train" else "Test"
        print(
            f" + {label} -> "
            f"MAE: {st['mae'][0]:.4f} ± {st['mae'][1]:.4f}\t"
            f"RMSD: {st['rmsd'][0]:.4f} ± {st['rmsd'][1]:.4f}\t"
            f"Max: {st['max'][0]:.4f} ± {st['max'][1]:.4f}\t"
            f"Min: {st['min'][0]:.4f} ± {st['min'][1]:.4f}\t"
            f"R^2: {st['r2'][0]:.4f} ± {st['r2'][1]:.4f}"
        )

In [ ]:
!pip install fvcore

In [ ]:
train_path = model_dataset_dir + f"/PAC5-4/fold_{fold}_train.csv"
val_path = model_dataset_dir + f"/PAC5-4/fold_{fold}_val.csv"
train_dataset = PAC5MatrixDataset(
    train_path, properties=target_properties, num_workers=4
)
val_dataset = PAC5MatrixDataset(val_path, properties=target_properties, num_workers=4)
scaler = MultiTargetScaler()
scaler.fit(train_dataset.targets.numpy())
train_dataset.targets = torch.tensor(
    scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32
)
val_dataset.targets = torch.tensor(
    scaler.transform(val_dataset.targets.numpy()), dtype=torch.float32
)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

import numpy as np
import torch
from fvcore.nn import FlopCountAnalysis

# 1. Instantiate model and put in evaluation mode
model = GroupEquivariantCNN(
    property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=32
).to(device)
model.eval()

# 2. Extract sample input from val_loader
sample_batch = next(iter(val_loader))
# Unpack feature tensor if dataset returns tuple (inputs, targets)
sample_input = (
    sample_batch[0] if isinstance(sample_batch, (list, tuple)) else sample_batch
)
sample_input = sample_input.to(device)

# Prepare single-sample input for individual latency profiling
single_input = sample_input[:1]

# -----------------------------------------------------------------------------
# 1. Parameter Count
# -----------------------------------------------------------------------------
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params / 1e3:.2f} K ({total_params:,})")

# -----------------------------------------------------------------------------
# 2. FLOPs Calculation
# -----------------------------------------------------------------------------
flops_analysis = FlopCountAnalysis(model, single_input)
total_flops = flops_analysis.total()
print(f"FLOPs (per sample): {total_flops / 1e6:.2f} M ({total_flops:,} FLOPs)")

# -----------------------------------------------------------------------------
# 3. GPU Latency Measurement
# -----------------------------------------------------------------------------
warmup_iters = 50
measure_iters = 1000

# Warm-up phase to reach stable GPU clock speed
with torch.no_grad():
    for _ in range(warmup_iters):
        _ = model(single_input)

# CUDA timing events
starter = torch.cuda.Event(enable_timing=True)
ender = torch.cuda.Event(enable_timing=True)
timings = np.zeros(measure_iters)

with torch.no_grad():
    for i in range(measure_iters):
        starter.record()
        _ = model(single_input)
        ender.record()

        torch.cuda.synchronize()  # Wait for GPU completion
        timings[i] = starter.elapsed_time(ender)

mean_latency = np.mean(timings)
std_latency = np.std(timings)

print(f"Inference latency (batch size = 1): {mean_latency:.3f} ± {std_latency:.3f} ms")